# Gradient Descent Regression

In the previous notebook, we used gradient descent as one method to solve linear regression. Here, we take a deeper dive into gradient descent itself - exploring how different variants (batch, stochastic, mini-batch) and hyperparameters (learning rate, iterations) affect convergence and prediction quality.

As covered in [CMOR 438 Lecture 4.1 (Gradient Descent)](../../Data_Science_and_Machine_Learning_Spring_2022/Lecture_4/Lecture_4_1_Gradient_Descent.ipynb), the gradient is a direction vector that points toward the steepest ascent of a function. By moving in the opposite direction, we minimize our cost function. [Lecture 4.2 (Regression Neuron)](../../Data_Science_and_Machine_Learning_Spring_2022/Lecture_4/Lecture_4_2_Regression_Neuron.ipynb) demonstrates this concept applied to a single neuron regression model using both batch and stochastic gradient descent.

We continue using the [California Housing Dataset](https://www.kaggle.com/datasets/camnugent/california-housing-prices) to predict median house values.

---

## Data Preprocessing
We apply the same preprocessing pipeline from the linear regression notebook: handle missing values, one-hot encode categorical features, z-score normalize, and split into train/test sets.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn as sk
from sklearn.model_selection import train_test_split

data_df = pd.read_csv("housing.csv")
data_df['total_bedrooms'] = data_df['total_bedrooms'].fillna(data_df['total_bedrooms'].median())
data_encoded = pd.get_dummies(data_df, columns=['ocean_proximity'], dtype=float)

data = data_encoded.to_numpy()
target_col = data_encoded.columns.get_loc('median_house_value')
feature_cols = [i for i in range(data.shape[1]) if i != target_col]
x_data = data[:, feature_cols]
y = data[:, target_col]
feature_names = [data_encoded.columns[i] for i in feature_cols]

mu = x_data.mean(axis=0)
sigma = x_data.std(axis=0)
sigma[sigma == 0] = 1
r_data = (x_data - mu) / sigma
r_data = np.c_[np.ones(r_data.shape[0]), r_data]

X_train, X_test, y_train, y_test = train_test_split(
    r_data, y, test_size=0.33, shuffle=True, random_state=42
)

print(f"Training: {X_train.shape}, Test: {X_test.shape}")

---

## Learning Rate Analysis

The learning rate $\alpha$ is the most critical hyperparameter in gradient descent. As discussed in CMOR 438 Lecture 4.1, setting $\alpha$ too high causes the algorithm to overshoot the minimum and diverge, while setting it too low results in extremely slow convergence.

We test multiple learning rates to visualize their effect on the training MSE curve.

In [ ]:
alphas = [0.001, 0.01, 0.05, 0.1, 0.3]
iterations = 500

fig, axs = plt.subplots(1, 2, figsize=(20, 6))

for alpha in alphas:
    theta = np.random.randn(X_train.shape[1]) * 0.01
    mse_history = []

    for i in range(iterations):
        y_hat = X_train @ theta
        mse = (1 / X_train.shape[0]) * np.sum((y_hat - y_train) ** 2)
        mse_history.append(mse)
        delta = (2 / X_train.shape[0]) * (X_train.T @ (y_hat - y_train))
        theta = theta - alpha * delta

    axs[0].plot(mse_history, label=f'α={alpha}')
    axs[1].plot(mse_history[:50], label=f'α={alpha}')

axs[0].set_xlabel('Iteration', fontsize=14)
axs[0].set_ylabel('MSE', fontsize=14)
axs[0].set_title('Training MSE vs Iteration (Full)', fontsize=16)
axs[0].legend(fontsize=12)
axs[0].set_ylim(0, 3e10)

axs[1].set_xlabel('Iteration', fontsize=14)
axs[1].set_ylabel('MSE', fontsize=14)
axs[1].set_title('Training MSE vs Iteration (First 50)', fontsize=16)
axs[1].legend(fontsize=12)
axs[1].set_ylim(0, 3e10)

plt.tight_layout()
plt.show()

The plots show a clear tradeoff: higher learning rates converge faster but risk instability, while lower rates are stable but slow. An $\alpha$ of 0.05-0.1 provides a good balance for this dataset.

---

## Gradient Descent Variants

As introduced in CMOR 438 Lecture 4.2, there are three main variants of gradient descent:

### Batch Gradient Descent
Uses the entire training set to compute the gradient at each step. Stable but slow for large datasets.

$\nabla\theta = \frac{2}{n} \sum_{i=1}^{n} X_i^T(\hat{y}_i - y_i)$

### Stochastic Gradient Descent (SGD)
Uses a single random sample per step. Fast but noisy - the loss curve oscillates rather than smoothly decreasing.

$\nabla\theta = 2 \cdot X_i^T(\hat{y}_i - y_i)$ for a random sample $i$

### Mini-Batch Gradient Descent
Uses a small batch of samples (e.g., 32 or 64). Balances the stability of batch with the speed of SGD.

In [ ]:
np.random.seed(42)
alpha = 0.05
iterations = 300

theta_batch = np.random.randn(X_train.shape[1]) * 0.01
theta_sgd = theta_batch.copy()
theta_mini = theta_batch.copy()

batch_mse = []
sgd_mse = []
mini_mse = []
batch_size = 64

for i in range(iterations):
    y_hat = X_train @ theta_batch
    batch_mse.append((1 / X_train.shape[0]) * np.sum((y_hat - y_train) ** 2))
    delta = (2 / X_train.shape[0]) * (X_train.T @ (y_hat - y_train))
    theta_batch = theta_batch - alpha * delta

    idx = np.random.randint(0, X_train.shape[0])
    x_i = X_train[idx:idx+1]
    y_i = y_train[idx:idx+1]
    y_hat_i = x_i @ theta_sgd
    sgd_mse.append((1 / X_train.shape[0]) * np.sum((X_train @ theta_sgd - y_train) ** 2))
    delta_sgd = 2 * (x_i.T @ (y_hat_i - y_i))
    theta_sgd = theta_sgd - alpha * delta_sgd

    indices = np.random.choice(X_train.shape[0], batch_size, replace=False)
    x_mb = X_train[indices]
    y_mb = y_train[indices]
    y_hat_mb = x_mb @ theta_mini
    mini_mse.append((1 / X_train.shape[0]) * np.sum((X_train @ theta_mini - y_train) ** 2))
    delta_mb = (2 / batch_size) * (x_mb.T @ (y_hat_mb - y_mb))
    theta_mini = theta_mini - alpha * delta_mb

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(batch_mse, label='Batch GD', linewidth=2)
ax.plot(sgd_mse, label='Stochastic GD', alpha=0.7)
ax.plot(mini_mse, label='Mini-Batch GD (64)', alpha=0.7)
ax.set_xlabel('Iteration', fontsize=14)
ax.set_ylabel('MSE', fontsize=14)
ax.set_title('Gradient Descent Variants Comparison', fontsize=16)
ax.legend(fontsize=12)
ax.set_ylim(0, 2e10)
plt.tight_layout()
plt.show()

Batch gradient descent shows the smoothest convergence curve. SGD is noisy because each update is based on a single sample, so the gradient estimate has high variance. Mini-batch strikes a middle ground - smoother than SGD but with some oscillation compared to full batch.

---

## Test Set Evaluation

We evaluate all three variants on the test set and compare their final performance.

In [ ]:
variants = {
    'Batch GD': theta_batch,
    'Stochastic GD': theta_sgd,
    'Mini-Batch GD': theta_mini
}

fig, axs = plt.subplots(3, 1, figsize=(18, 18))

for i, (name, theta_v) in enumerate(variants.items()):
    y_pred = X_test @ theta_v

    mse = np.round((1 / X_test.shape[0]) * np.sum((y_pred - y_test) ** 2), 2)
    mae = np.round((1 / X_test.shape[0]) * np.sum(np.abs(y_pred - y_test)), 2)
    r2 = np.round(sk.metrics.r2_score(y_test, y_pred), 4)

    axs[i].scatter(range(X_test.shape[0]), y_pred, s=5, alpha=0.3, label='Predicted')
    axs[i].scatter(range(X_test.shape[0]), y_test, s=5, alpha=0.3, label='Actual')
    axs[i].set_title(name, fontsize=16)
    axs[i].set_xlabel('Sample Number', fontsize=12)
    axs[i].set_ylabel('House Value ($)', fontsize=12)
    axs[i].legend(fontsize=10)

    errors = f"MSE: {mse}\nMAE: {mae}\nR2: {r2}"
    axs[i].text(0.01, 0.95, errors, transform=axs[i].transAxes, verticalalignment='top',
               fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

---

## Error Distribution

Let's look at how the prediction errors are distributed for each variant.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(20, 5))

for i, (name, theta_v) in enumerate(variants.items()):
    errors = X_test @ theta_v - y_test
    axs[i].hist(errors, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
    axs[i].set_title(f'{name} Error Distribution', fontsize=14)
    axs[i].set_xlabel('Prediction Error ($)', fontsize=12)
    axs[i].set_ylabel('Frequency', fontsize=12)
    axs[i].axvline(x=0, color='red', linestyle='--', linewidth=2)

plt.tight_layout()
plt.show()

All three variants produce roughly normally distributed errors centered near zero, which is expected for a well-behaved linear model. The long right tail reflects the model's difficulty with high-value properties and the $500,001 cap in the dataset.

---

## Summary

Gradient descent is the foundational optimization algorithm behind most machine learning models. For this dataset:
- **Batch GD** gives the most stable convergence and best final MSE
- **SGD** converges quickly per-step but oscillates, and with only 300 iterations per sample it underfits compared to batch
- **Mini-batch** provides a practical compromise used in most real-world deep learning

The learning rate is the single most important hyperparameter - too high and the model diverges, too low and it never converges. For this dataset, $\alpha \approx 0.05$ works well across all variants.

All variants achieve similar final performance to the normal equation solution from the linear regression notebook, confirming that gradient descent finds the same minimum given enough iterations.